In [17]:
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
base_path = r"C:\Users\jmpar\OneDrive\Desktop\Dhvani\Retail Sales & Inventory Sys\Retail Sales Cleaned Data - dataset\archive"
db_path = "retail_sales.db"

In [ ]:
# 2. Load CSV files into pandas DataFrames

orders      = pd.read_csv(base_path / "orders.csv")
order_items = pd.read_csv(base_path / "order_items.csv")
customers   = pd.read_csv(base_path / "customers.csv")
stores      = pd.read_csv(base_path / "stores.csv")
staffs      = pd.read_csv(base_path / "staffs.csv")
products    = pd.read_csv(base_path / "products.csv")
stocks      = pd.read_csv(base_path / "stocks.csv")
brands      = pd.read_csv(base_path / "brands.csv")
categories  = pd.read_csv(base_path / "categories.csv")

print("All files loaded successfully!")
print("Orders shape:", orders.shape)

All files loaded successfully!
Orders shape: (1615, 8)


In [22]:
conn = sqlite3.connect(db_path)

In [ ]:
# Save each DataFrame as SQL table

orders.to_sql("orders", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)
stores.to_sql("stores", conn, if_exists="replace", index=False)
staffs.to_sql("staffs", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
stocks.to_sql("stocks", conn, if_exists="replace", index=False)
brands.to_sql("brands", conn, if_exists="replace", index=False)
categories.to_sql("categories", conn, if_exists="replace", index=False)

print("Saved tables to", db_path)

Saved tables to retail_sales.db


In [ ]:
# Integrity checks

q1 = "SELECT o.order_id FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id WHERE c.customer_id IS NULL;"
q2 = "SELECT oi.item_id FROM order_items oi LEFT JOIN orders o ON oi.order_id = o.order_id WHERE o.order_id IS NULL;"
q3 = "SELECT oi.item_id FROM order_items oi LEFT JOIN products p ON oi.product_id = p.product_id WHERE p.product_id IS NULL;"
q4 = "SELECT p.product_id FROM products p LEFT JOIN stocks s ON p.product_id = s.product_id WHERE s.product_id IS NULL;"

print("Orders with missing customers:")
display(pd.read_sql_query(q1, conn).head())
print("Order_items with missing orders:")
display(pd.read_sql_query(q2, conn).head())
print("Order_items with missing products:")
display(pd.read_sql_query(q3, conn).head())
print("Products missing stock row:")
display(pd.read_sql_query(q4, conn).head()) 

Orders with missing customers:


,order_id


Order_items with missing orders:


,item_id


Order_items with missing products:


,item_id


Products missing stock row:


,product_id
0,314
1,315
2,316
3,317
4,318


In [ ]:
# 5. Create views 
conn.execute("""
CREATE VIEW IF NOT EXISTS v_store_sales AS
SELECT s.store_id, s.store_name, s.city, s.state,
       SUM(oi.quantity * oi.list_price) AS total_sales
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN stores s ON o.store_id = s.store_id
GROUP BY s.store_id, s.store_name, s.city, s.state;
""")

conn.execute("""
CREATE VIEW IF NOT EXISTS v_product_sales AS
SELECT p.product_id, p.product_name,
       COALESCE(b.brand_name,'') AS brand_name,
       COALESCE(c.category_name,'') AS category_name,
       SUM(oi.quantity) AS total_units_sold,
       SUM(oi.quantity * oi.list_price) AS total_sales,
       COALESCE(st.quantity,0) AS current_stock
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN brands b ON p.brand_id = b.brand_id
LEFT JOIN categories c ON p.category_id = c.category_id
LEFT JOIN stocks st ON p.product_id = st.product_id
GROUP BY p.product_id, p.product_name, b.brand_name, c.category_name, st.quantity;
""")

conn.commit()
print("Views created: v_store_sales, v_product_sales") 

Views created: v_store_sales, v_product_sales


In [ ]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

          name
0       orders
1  order_items
2    customers
3       stores
4       staffs
5     products
6       stocks
7       brands
8   categories


#### Store-wise and State-wise Sales Analysis
Insight: Finds which stores (and which states) have the highest total sales.

In [ ]:
df_store = pd.read_sql_query("SELECT * FROM v_store_sales ORDER BY total_sales DESC LIMIT 20;", conn)
display(df_store)

,store_name,city,state,total_sales
0,Baldwin Bikes,Baldwin,NY,5826242.21
1,Santa Cruz Bikes,Santa Cruz,CA,1790145.91
2,Rowlett Bikes,Rowlett,TX,962600.76


#### Product-wise Sales and Inventory Trends
Insight: Shows best-selling products, their brands, categories, and remaining stock.

In [ ]:
product_trends = pd.read_sql_query("""
SELECT 
    p.product_name,
    b.brand_name,
    c.category_name,
    SUM(oi.quantity) AS total_units_sold,
    SUM(oi.quantity * oi.list_price) AS total_sales,
    st.quantity AS current_stock
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN brands b ON p.brand_id = b.brand_id
JOIN categories c ON p.category_id = c.category_id
JOIN stocks st ON p.product_id = st.product_id
GROUP BY p.product_name, b.brand_name, c.category_name
ORDER BY total_sales DESC;
""", conn)

product_trends.head()

,product_name,brand_name,category_name,total_units_sold,total_sales,current_stock
0,Trek Slash 8 27.5 - 2016,Trek,Mountain Bikes,462,1847995.38,8
1,Trek Conduit+ - 2016,Trek,Electric Bikes,435,1304995.65,11
2,Trek Fuel EX 8 29 - 2016,Trek,Mountain Bikes,429,1244095.71,2
3,Surly Straggler 650b - 2016,Surly,Cyclocross Bicycles,453,761488.47,8
4,Trek Domane SLR 6 Disc - 2017,Trek,Road Bikes,129,709498.71,15


#### Top 20 products by revenue

In [34]:
# top 20 products by revenue
df_prod = pd.read_sql_query("SELECT * FROM v_product_sales ORDER BY total_sales DESC LIMIT 20;", conn)
display(df_prod)

,product_id,product_name,brand_name,category_name,total_units_sold,total_sales,current_stock
0,7,Trek Slash 8 27.5 - 2016,Trek,Mountain Bikes,308,1231996.92,8
1,7,Trek Slash 8 27.5 - 2016,Trek,Mountain Bikes,154,615998.46,12
2,9,Trek Conduit+ - 2016,Trek,Electric Bikes,145,434998.55,11
3,9,Trek Conduit+ - 2016,Trek,Electric Bikes,145,434998.55,17
4,9,Trek Conduit+ - 2016,Trek,Electric Bikes,145,434998.55,23
5,4,Trek Fuel EX 8 29 - 2016,Trek,Mountain Bikes,143,414698.57,2
6,4,Trek Fuel EX 8 29 - 2016,Trek,Mountain Bikes,143,414698.57,11
7,4,Trek Fuel EX 8 29 - 2016,Trek,Mountain Bikes,143,414698.57,23
8,51,Trek Silque SLR 8 Women's - 2017,Trek,Road Bikes,58,376999.42,2
9,43,Trek Fuel EX 9.8 27.5 Plus - 2017,Trek,Mountain Bikes,66,349799.34,2


#### Staff Performance Reports
Insight: Evaluates staff by the total sales they processed.

In [ ]:
df_staff = pd.read_sql_query("""
SELECT sf.staff_id, sf.first_name || ' ' || sf.last_name AS staff_name, s.store_name,
       SUM(oi.quantity * oi.list_price) AS total_sales
FROM staffs sf
JOIN orders o ON o.staff_id = sf.staff_id
JOIN order_items oi ON oi.order_id = o.order_id
JOIN stores s ON o.store_id = s.store_id
GROUP BY sf.staff_id, staff_name, s.store_name
ORDER BY total_sales DESC LIMIT 20;
""", conn)
display(df_staff)       

,staff_id,staff_name,store_name,total_sales
0,6,Marcelene Boyer,Baldwin Bikes,2938888.73
1,7,Venita Daniel,Baldwin Bikes,2887353.48
2,3,Genna Serrano,Santa Cruz Bikes,952722.26
3,2,Mireya Copeland,Santa Cruz Bikes,837423.65
4,8,Kali Vargas,Rowlett Bikes,516695.17
5,9,Layla Terrell,Rowlett Bikes,445905.59


#### Customer Orders and Order Frequency
Insight: Tracks how often customers order and how much they spend.

In [ ]:
customer_orders = pd.read_sql_query("""
SELECT 
    c.first_name || ' ' || c.last_name AS customer_name,
    COUNT(o.order_id) AS total_orders,
    SUM(oi.quantity * oi.list_price) AS total_spent
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY customer_name
ORDER BY total_orders DESC;
""", conn)

customer_orders.head()

,customer_name,total_orders,total_spent
0,Tameka Fisher,13,26249.81
1,Emmitt Sanchez,12,34503.82
2,Pamelia Newman,11,37801.84
3,Mozelle Carter,11,25382.84
4,Lyndsey Bean,11,35857.86


#### Revenue and Discount Analysis
Insight: Tracks monthly revenue and discount trends.

In [ ]:
revenue_discount = pd.read_sql_query("""
SELECT 
    STRFTIME('%Y-%m', o.order_date) AS order_month,
    SUM(oi.quantity * oi.list_price) AS gross_revenue,
    SUM(oi.discount * oi.quantity) AS total_discount,
    (SUM(oi.quantity * oi.list_price) - SUM(oi.discount * oi.quantity)) AS net_revenue
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
GROUP BY order_month
ORDER BY order_month;
""", conn)

revenue_discount.head()

,order_month,gross_revenue,total_discount,net_revenue
0,2016-01,241184.15,23.04,241161.11
1,2016-02,175768.10,25.01,175743.09
2,2016-03,202157.14,22.28,202134.86
3,2016-04,187223.55,18.76,187204.79
4,2016-05,228701.13,22.34,228678.79


#### Monthly Revenue
Insight: Net revenue varies monthly, showing sales trends and the impact of discounts on overall earnings.

In [ ]:
df_month = pd.read_sql_query("""
SELECT STRFTIME('%Y-%m', o.order_date) AS order_month,
       SUM(oi.quantity * oi.list_price) AS gross_revenue,
       SUM(COALESCE(oi.discount,0) * oi.quantity) AS total_discount,
       SUM(oi.quantity * oi.list_price) - SUM(COALESCE(oi.discount,0) * oi.quantity) AS net_revenue
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
GROUP BY order_month
ORDER BY order_month;
""", conn)
display(df_month) 

,order_month,gross_revenue,total_discount,net_revenue
0,2016-01,241184.15,23.04,241161.11
1,2016-02,175768.10,25.01,175743.09
2,2016-03,202157.14,22.28,202134.86
3,2016-04,187223.55,18.76,187204.79
4,2016-05,228701.13,22.34,228678.79
5,2016-06,231120.29,19.91,231100.38
6,2016-07,222854.21,23.32,222830.89
7,2016-08,253130.83,25.17,253105.66
8,2016-09,303282.61,28.79,303253.82
9,2016-10,235051.79,25.07,235026.72


In [ ]:
# Load customer orders view and compute simple RFM
df_customers = pd.read_sql_query("SELECT * FROM v_customer_orders;", conn) \
               if 'v_customer_orders' in [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='view';")] \
               else None

# If view doesn't exist, create it quickly then reload
if df_customers is None:
    conn.execute("""
    CREATE VIEW IF NOT EXISTS v_customer_orders AS
    SELECT c.customer_id, c.first_name || ' ' || c.last_name AS customer_name,
           COUNT(DISTINCT o.order_id) AS total_orders,
           SUM(oi.quantity * oi.list_price) AS total_spent,
           MAX(o.order_date) AS last_order_date
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    LEFT JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_id, customer_name;
    """)
    conn.commit()
    df_customers = pd.read_sql_query("SELECT * FROM v_customer_orders;", conn)

# compute RFM
df_customers['last_order_date'] = pd.to_datetime(df_customers['last_order_date'])
today = pd.to_datetime('today').normalize()
df_customers['recency_days'] = (today - df_customers['last_order_date']).dt.days.fillna(9999).astype(int)
df_customers['frequency'] = df_customers['total_orders'].fillna(0).astype(int)
df_customers['monetary'] = df_customers['total_spent'].fillna(0.0)

display(df_customers.head())

,customer_id,customer_name,total_orders,total_spent,last_order_date,recency_days,frequency,monetary
0,1,Debra Burks,3,30645.87,2018-11-18,2526,3,30645.87
1,2,Kasha Todd,3,21653.85,2018-04-09,2749,3,21653.85
2,3,Tameka Fisher,3,26249.81,2018-10-21,2554,3,26249.81
3,4,Daryl Spence,3,24198.88,2018-04-18,2740,3,24198.88
4,5,Charolette Rice,3,19442.88,2018-04-17,2741,3,19442.88


In [ ]:
# run KMeans clustering on RFM

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np

rfm = df_customers[['recency_days','frequency','monetary']].fillna(0)
if len(rfm) >= 5 and len(rfm.drop_duplicates()) > 1:
    scaler = StandardScaler()
    Xs = scaler.fit_transform(rfm)
    k = 4
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(Xs)
    df_customers['segment'] = labels
    print("Assigned segments 0..", k-1)
else:
    df_customers['segment'] = 0
    print("Too few customers – all assigned to segment 0")

display(df_customers[['customer_id','customer_name','recency_days','frequency','monetary','segment']].head())

Assigned segments 0.. 3


,customer_id,customer_name,recency_days,frequency,monetary,segment
0,1,Debra Burks,2526,3,30645.87,2
1,2,Kasha Todd,2749,3,21653.85,2
2,3,Tameka Fisher,2554,3,26249.81,2
3,4,Daryl Spence,2740,3,24198.88,2
4,5,Charolette Rice,2741,3,19442.88,2
